# Document Classification Example

Evaluates LLMs on 16-class document classification using [RVL-CDIP](https://huggingface.co/datasets/dvgodoy/rvl_cdip_mini) (OCR-extracted text).

Uses a **QuestionPipeline**: **TemplateQuestionGenerator** → **QuestionRenderer** → **RolloutGenerator**, then scores with `compute_metrics_summary()`.

In [1]:
%pip install lightningrod-ai python-dotenv datasets pandas

from IPython.display import clear_output
clear_output()

## Set up the client

Sign up at [dashboard.lightningrod.ai](https://dashboard.lightningrod.ai/?redirect=/api) to get your API key and **$50 of free credits**.

- **Google Colab**: Go to the Secrets section (key icon in left sidebar) and add a secret named `LIGHTNINGROD_API_KEY`
- **Local Jupyter**: Set the `LIGHTNINGROD_API_KEY` environment variable, or you'll be prompted to enter it

In [4]:
from dotenv import load_dotenv
from lightningrod import LightningRod
from lightningrod.utils import config

load_dotenv()

api_key = config.get_config_value("LIGHTNINGROD_API_KEY")
lr = LightningRod(api_key=api_key)


## Download and prepare benchmark data

We use the [dvgodoy/rvl_cdip_mini](https://huggingface.co/datasets/dvgodoy/rvl_cdip_mini) dataset from HuggingFace — a 4,000-document subset of the RVL-CDIP benchmark with 16 document types and pre-extracted OCR text.

Each document is converted to a `Sample` with the OCR text as the seed and the ground-truth label stored for later evaluation.

Alternatively, you can use a local CSV file with `text` and `label` columns. Use `file_to_samples` from `lightningrod.preprocessing` — it creates one sample per row. Standard columns are `text`, `seed_text`, or `content` for the seed; `label` for the ground-truth. Override with `csv_text_column` and `csv_label_column` if your CSV uses different headers.

In [ ]:
import random
from datasets import load_dataset
from lightningrod import create_sample
from typing import List, Tuple

LABEL_NAMES = [
    "letter", "form", "email", "handwritten", "advertisement",
    "scientific_report", "scientific_publication", "specification",
    "file_folder", "news_article", "budget", "invoice",
    "presentation", "questionnaire", "resume", "memo",
]

MIN_TEXT_LENGTH = 50
MAX_DOC_LENGTH = 4000
NUM_SAMPLES = 20

ds = load_dataset("dvgodoy/rvl_cdip_mini", split="test")

# Keep only non-image columns
if "image" in ds.column_names:
    ds = ds.remove_columns(["image"])
    
text_and_labels: List[Tuple[str, str]] = []
for ex in ds:
    paragraphs = ex.get("ocr_paragraphs") or []
    doc_text = "\n\n".join(paragraphs).strip()
    if len(doc_text) < MIN_TEXT_LENGTH:
        continue
    if len(doc_text) > MAX_DOC_LENGTH:
        doc_text = doc_text[:MAX_DOC_LENGTH] + "\n... (truncated)"

    label_name = LABEL_NAMES[ex["label"]]
    text_and_labels.append((doc_text, label_name))

# Construct Samples:
samples = []
for (text, label) in text_and_labels:
    samples.append(create_sample(seed_text=text, label=label))

random.seed(42)
samples = random.sample(samples, min(NUM_SAMPLES, len(samples)))
print(f"{len(samples)} samples ready for evaluation")

## Alternative: Load from local CSV

Instead of HuggingFace, you can also load samples from a local CSV. Set `LOCAL_CSV_FILE_PATH` to the path of your local file with `seed_text` and `label` columns (can be customized).

In [ ]:
LOCAL_CSV_FILE_PATH = config.get_config_value("LOCAL_CSV_FILE_PATH")
if LOCAL_CSV_FILE_PATH:
    from lightningrod.preprocessing import file_to_samples

    samples = file_to_samples(LOCAL_CSV_FILE_PATH, csv_seed_text_col="seed_text", csv_label_col="label")
    LABEL_NAMES = sorted(set(s.label.label for s in samples if s.label))
    OPTIONS = {f"option_{i}": name for i, name in enumerate(LABEL_NAMES)}
    print(f"{len(samples)} samples ready for evaluation")

## Upload input dataset

In [6]:
input_dataset = lr.datasets.create_from_samples(samples)
print(f"Created input dataset: {input_dataset.id}")
print(f"Total samples: {input_dataset.num_rows}")

Created input dataset: c62082b8-8260-4ff5-a573-24dbaa094fa8
Total samples: 20


## Configure the pipeline

The pipeline has three stages:
1. **TemplateQuestionGenerator** — fills a classification prompt template with each document's OCR text
2. **QuestionRenderer** — renders the question with a multiple-choice answer type
3. **RolloutGenerator** — sends the rendered prompt to multiple LLMs via OpenRouter

In [ ]:
from lightningrod import (
    QuestionPipeline,
    TemplateQuestionGenerator,
    QuestionRenderer,
    RolloutGenerator,
    MultipleChoiceAnswerType,
    ModelConfig,
    ModelSourceType,
    RolloutScorer,
)

OPTIONS = {f"option_{i}": name for i, name in enumerate(LABEL_NAMES)}

_options_display = "\n".join(f"{chr(65 + i)}) {name}" for i, name in enumerate(LABEL_NAMES))
QUESTION_TEMPLATE = (
    "What type of document is this? Classify it as one of the following categories:\n\n"
    f"{_options_display}\n\n"
    "Document text:\n{seed_text}"
)

models = [
    ModelConfig(model_name="openai/gpt-4.1-mini", model_source=ModelSourceType.OPEN_ROUTER, use_pipeline_key=True),
    ModelConfig(model_name="anthropic/claude-sonnet-4", model_source=ModelSourceType.OPEN_ROUTER, use_pipeline_key=True),
    ModelConfig(model_name="google/gemini-2.5-flash", model_source=ModelSourceType.OPEN_ROUTER, use_pipeline_key=True),
]

answer_type = MultipleChoiceAnswerType()

pipeline = QuestionPipeline(
    question_generator=TemplateQuestionGenerator(question_template=QUESTION_TEMPLATE),
    renderer=QuestionRenderer(answer_type=answer_type),
    rollout_generator=RolloutGenerator(models=models),
    scorer=RolloutScorer(answer_type=answer_type, multiple_choice_options=OPTIONS),
)

## Run the pipeline

This sends each document to all three models for classification. It may take a few minutes depending on the number of samples.

In [ ]:
dataset = lr.transforms.run(
    pipeline,
    input_dataset=input_dataset,
    name="Document Classification",
)

## View results

Download the results and compute per-model accuracy using `compute_metrics_summary()`.

In [9]:
import pandas as pd
from lightningrod.utils import compute_metrics_summary

result_samples = dataset.download()

summary = compute_metrics_summary(result_samples, OPTIONS)
df = pd.DataFrame.from_dict(summary, orient="index")
df.index.name = "model"
df

,accuracy,n_correct,n_parsed,n_total,parse_rate,mean_reward
model,,,,,,
openai/gpt-4.1-mini,0.4,8,20,20,1.00,-2.211777
anthropic/claude-sonnet-4,0.7,14,20,20,1.00,-1.393639
google/gemini-2.5-flash,0.6,9,15,20,0.75,-3.746457
